# 6. Least squares for overdetermined systems  [1.8.2]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_triangular
from ipywidgets import interact, Checkbox, fixed

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 6.1 Fit a line, see the geometry

**Task.** Generate 30 noisy points $y = 2 + 0.5t + \text{noise}$. Fit a line by solving the normal equations. Plot the data, the fitted line and the vertical residuals.

$$A = \begin{bmatrix}1 & t_1\\ \vdots & \vdots\\ 1 & t_{30}\end{bmatrix}, \qquad A^\top A\hat x = A^\top y, \qquad e = y - A\hat x.$$

### Prediction (write this *before* running)

- How close will $\hat x$ be to $(2, 0.5)$?
- What is $\sum_i e_i$, the sum of the residuals?

*(fill in)*

In [ ]:
rng = np.random.default_rng(0)
t = np.sort(rng.uniform(0, 10, 30))
y = 2 + 0.5 * t + rng.normal(0, 1.0, t.size)

A = np.column_stack([np.ones_like(t), t])
x_hat = np.linalg.solve(A.T @ A, A.T @ y)          # normal equations
y_fit = A @ x_hat
e = y - y_fit
print("x̂ =", x_hat, "  (true: [2, 0.5])")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.vlines(t, y_fit, y, color="crimson", lw=1.2, label="residuals e")
ax.plot(t, y, "ko", ms=5, label="data")
tt = np.linspace(0, 10, 2)
ax.plot(tt, x_hat[0] + x_hat[1] * tt, "royalblue", lw=2, label=f"fit: {x_hat[0]:.3f} + {x_hat[1]:.3f} t")
ax.plot(tt, 2 + 0.5 * tt, "k--", lw=1, label="true: 2 + 0.5 t")
ax.set_xlabel("t"); ax.set_ylabel("y"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

### Verify numerically

In [ ]:
U, s, Vt = np.linalg.svd(A)
P_CS = U[:, :2] @ U[:, :2].T

check("Aᵀ e = 0", np.allclose(A.T @ e, 0))
check("  first row: Σ eᵢ = 0", np.isclose(e.sum(), 0))
check("  second row: Σ tᵢ eᵢ = 0", np.isclose(t @ e, 0))
check("e is in LNS(A): its projection onto CS(A) is 0", np.allclose(P_CS @ e, 0))
check("same answer as np.linalg.lstsq", np.allclose(x_hat, np.linalg.lstsq(A, y, rcond=None)[0]))
check("‖y‖² = ‖Ax̂‖² + ‖e‖²  (Pythagoras)", np.isclose(y @ y, y_fit @ y_fit + e @ e))
perturbed = [np.linalg.norm(y - A @ (x_hat + d)) for d in rng.normal(0, 0.05, (1000, 2))]
check("every one of 1000 random perturbations of x̂ gives a larger ‖e‖", min(perturbed) > np.linalg.norm(e))

---
### Explain (hand-in — in your own words)

**1. Verify $A^\top e = 0$. In which subspace of $A$ is $e$?**

*Your answer.*

**2. Why is that the condition for the smallest $\|e\|_2$ [1.8.8]?**
*(Hint: any other $x$ gives $y - Ax = e + A(\hat x - x)$. The two pieces are perpendicular, so use Pythagoras.)*

*Your answer.*

---
## 6.2 The error landscape

**Task.** Plot the cost below as a surface and contour map over $(x_1, x_2)$ for the previous problem, and mark the least-squares solution.

$$J(x) = \|Ax - y\|_2^2, \qquad \nabla J = 2A^\top(Ax - y), \qquad \nabla^2 J = 2A^\top A.$$

The arrows at $\hat x$ are the eigenvectors $q_i$ of $A^\top A$, scaled by $\sqrt{J(\hat x)/\lambda_i}$, so they end exactly on the contour $J = 2J(\hat x)$. Both axes use the same scale, so the shape you see is the true shape.

### Prediction (write this *before* running)

- Will the contours be circles, or long thin ellipses? Which direction is the long one?

*(fill in)*

In [ ]:
def cost_landscape(t, y, center=False):
    tt = t - t.mean() if center else t
    A = np.column_stack([np.ones_like(tt), tt])
    AtA = A.T @ A
    x_hat = np.linalg.solve(AtA, A.T @ y)
    J_hat = np.sum((A @ x_hat - y) ** 2)
    lam, Q = np.linalg.eigh(AtA)

    half = 1.3 * np.sqrt(J_hat / lam.min())               # box that contains the J = 2 J(x̂) contour
    g1 = np.linspace(x_hat[0] - half, x_hat[0] + half, 151)
    g2 = np.linspace(x_hat[1] - half, x_hat[1] + half, 151)
    X1, X2 = np.meshgrid(g1, g2)
    R = A[:, 0, None, None] * X1 + A[:, 1, None, None] * X2 - y[:, None, None]
    J = np.sum(R**2, axis=0)

    fig = plt.figure(figsize=(15, 6))
    ax3 = fig.add_subplot(1, 2, 1, projection="3d")
    ax3.plot_surface(X1, X2, J, cmap="viridis", alpha=0.85, linewidth=0)
    ax3.scatter(*x_hat, J_hat, color="red", s=50)
    ax3.set_xlabel("x₁ (intercept)"); ax3.set_ylabel("x₂ (slope)"); ax3.set_zlabel("J")

    ax = fig.add_subplot(1, 2, 2)
    levels = J_hat * np.array([1.05, 1.2, 1.5, 2, 3, 5, 8])
    cs = ax.contour(X1, X2, J, levels=levels, cmap="viridis")
    ax.clabel(cs, fontsize=7, fmt="%.0f")
    ax.plot(*x_hat, "r*", ms=15, label="x̂")
    for i, col in enumerate(["crimson", "royalblue"]):
        tip = x_hat + np.sqrt(J_hat / lam[i]) * Q[:, i]
        ax.annotate("", xy=tip, xytext=x_hat, arrowprops=dict(arrowstyle="->", color=col, lw=2))
        ax.text(*tip, f" λ{i+1} = {lam[i]:.1f}", color=col)
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
    ax.set_xlabel("x₁ (intercept)"); ax.set_ylabel("x₂ (slope)")
    ax.set_title(f"{'centred t' if center else 'raw t'}:  eig(AᵀA) = {lam.round(2)},  κ(AᵀA) = {lam.max() / lam.min():.1f}")
    plt.show()
    return A, x_hat

A2, x_hat2 = cost_landscape(t, y)

### Verify numerically

In [ ]:
J = lambda x: np.sum((A2 @ x - y) ** 2)
grad = lambda x: 2 * A2.T @ (A2 @ x - y)
h = 1e-4
H_fd = np.array([[(J(x_hat2 + h * (ei + ej)) - J(x_hat2 + h * ei) - J(x_hat2 + h * ej) + J(x_hat2)) / h**2
                  for ej in np.eye(2)] for ei in np.eye(2)])
d = np.random.default_rng(1).standard_normal(2)

check("∇J(x̂) = 0", np.allclose(grad(x_hat2), 0, atol=1e-9))
check("finite-difference Hessian = 2AᵀA", np.allclose(H_fd, 2 * A2.T @ A2, rtol=1e-3))
check("J(x̂ + d) − J(x̂) = dᵀ AᵀA d  (exact quadratic)", np.isclose(J(x_hat2 + d) - J(x_hat2), d @ A2.T @ A2 @ d))
check("AᵀA is positive definite", np.linalg.eigvalsh(A2.T @ A2).min() > 0)

### Explore: centre the inputs ($t \to t - \bar t$)

Predict first: what happens to $A^\top A$, its eigenvectors and the contours?

In [ ]:
@interact(center=Checkbox(value=True, description="centre t"), t=fixed(t), y=fixed(y))
def explore_centering(center, t, y):
    cost_landscape(t, y, center)

---
### Explain (hand-in — in your own words)

**1. Why is it a paraboloid?**
*(Hint: expand $\|Ax - y\|^2$. What kind of function of $x$ is it?)*

*Your answer.*

**2. What do the eigenvalues of $A^\top A$ say about the shape of the contours? (Link to Section 2.)**
*(Hint: around $\hat x$, $J(\hat x + d) = J(\hat x) + d^\top A^\top A d$, a quadratic form.)*

*Your answer.*

**3. Why does centring $t$ make the contours rounder and axis-aligned?**
*(Hint: look at the off-diagonal entry of $A^\top A$.)*

*Your answer.*

---
## 6.3 Reproduce Example 1.14

**Task.** Solve the system below with the normal equations, `np.linalg.lstsq`, the pseudoinverse and the QR decomposition.

$$\begin{bmatrix}2&1\\1&3\\0&1\end{bmatrix}\begin{bmatrix}x_1\\x_2\end{bmatrix} = \begin{bmatrix}3\\1\\4\end{bmatrix}, \qquad \hat x = (A^\top A)^{-1}A^\top b.$$

QR: with $A = QR$ ($Q$ has orthonormal columns, $R$ is upper triangular), the normal equations become $R\hat x = Q^\top b$.

### Prediction (write this *before* running)

- Is there an exact solution? Where will $\hat x$ sit relative to the three lines?

*(fill in)*

In [ ]:
A = np.array([[2., 1.], [1., 3.], [0., 1.]])
b = np.array([3., 1., 4.])

Q, R = np.linalg.qr(A)                                 # reduced QR: Q 3×2, R 2×2
solutions = {
    "normal equations": np.linalg.solve(A.T @ A, A.T @ b),
    "np.linalg.lstsq": np.linalg.lstsq(A, b, rcond=None)[0],
    "pseudoinverse": np.linalg.pinv(A) @ b,
    "QR": solve_triangular(R, Q.T @ b),
}
pd.DataFrame({k: {"x₁": v[0], "x₂": v[1], "‖Ax − b‖": np.linalg.norm(A @ v - b)} for k, v in solutions.items()}).T

In [ ]:
for name, x in solutions.items():
    check(f"{name}: x̂ = [0.9, 0.5]", np.allclose(x, [0.9, 0.5]))
check("(AᵀA)⁻¹ = [[11/30, −1/6], [−1/6, 1/6]]  (notes)", np.allclose(np.linalg.inv(A.T @ A), [[11/30, -1/6], [-1/6, 1/6]]))
check("QR: QᵀQ = I and R upper triangular", np.allclose(Q.T @ Q, np.eye(2)) and np.isclose(R[1, 0], 0))
check("no exact solution: Ax̂ ≠ b", not np.allclose(A @ solutions["QR"], b))

### The three lines and the least-squares point (Figure 4, right)

In [ ]:
x_hat = solutions["QR"]
g = np.linspace(-3, 4, 300)
X1, X2 = np.meshgrid(g, g)
Jgrid = sum((A[i, 0] * X1 + A[i, 1] * X2 - b[i]) ** 2 for i in range(3))

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.contourf(X1, X2, Jgrid, levels=30, cmap="Greys", alpha=0.35)
xx = np.linspace(-3, 4, 2)
for i, col in enumerate(["C0", "C1", "C2"]):
    a1, a2 = A[i]
    if abs(a2) > 1e-12:
        ax.plot(xx, (b[i] - a1 * xx) / a2, col, lw=2, label=f"{a1:g}x₁ + {a2:g}x₂ = {b[i]:g}")
    else:
        ax.axvline(b[i] / a1, color=col, lw=2, label=f"{a1:g}x₁ = {b[i]:g}")
for i, j in [(0, 1), (0, 2), (1, 2)]:                  # pairwise intersections
    p = np.linalg.solve(A[[i, j]], b[[i, j]])
    ax.plot(*p, "kx", ms=8)
ax.plot(*x_hat, "r*", ms=18, label=f"least squares x̂ = {x_hat.round(3)}")
ax.set_xlim(-3, 4); ax.set_ylim(-3, 4); ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax.legend(loc="lower left"); ax.set_xlabel("x₁"); ax.set_ylabel("x₂")
ax.set_title("three equations, two unknowns  (shading: ‖Ax − b‖²)")
plt.show()

---
### Explain (hand-in — in your own words)

**1. Do all four agree with $\hat x = [0.9, 0.5]^\top$?**

*Your answer.*

**2. Draw the three lines of the system in the plane and the least-squares point (Figure 4, right). Why does $\hat x$ not lie on any of the lines?**

*Your answer.*

---
## 6.4 When the normal equations go wrong

**Task.** Fit a degree-10 polynomial with $A$ built from columns $1, t, t^2, \dots, t^{10}$, and compare the normal-equations solution with `np.linalg.lstsq`.

$$\kappa(A^\top A) = \kappa(A)^2.$$

To measure the error exactly, we first use data generated by known coefficients: $y = Ac_{\text{true}}$, so $c_{\text{true}}$ is the exact answer.

### Prediction (write this *before* running)

- Roughly how big is $\kappa(A)$ for $t \in [0, 1]$ and degree 10?
- How many correct digits will each method give?

*(fill in)*

In [ ]:
rng = np.random.default_rng(0)
deg = 10
t = np.linspace(0, 1, 50)
A = np.vander(t, deg + 1, increasing=True)            # columns 1, t, ..., t^10
c_true = rng.standard_normal(deg + 1)
y = A @ c_true

c_ne = np.linalg.solve(A.T @ A, A.T @ y)
c_ls = np.linalg.lstsq(A, y, rcond=None)[0]

print(f"cond(A)      = {np.linalg.cond(A):.3e}")
print(f"cond(AᵀA)    = {np.linalg.cond(A.T @ A):.3e}")
print(f"cond(A)²     = {np.linalg.cond(A) ** 2:.3e}\n")
for name, c in [("normal equations", c_ne), ("lstsq", c_ls)]:
    print(f"{name:>17}: relative coefficient error {np.linalg.norm(c - c_true) / np.linalg.norm(c_true):.2e}"
          f"   residual ‖Ac − y‖ = {np.linalg.norm(A @ c - y):.2e}")

A realistic fit: noisy samples of $\sin(2\pi t)$. The curves can look alike while the coefficients do not.

In [ ]:
y_noisy = np.sin(2 * np.pi * t) + rng.normal(0, 0.05, t.size)
c_ne_n = np.linalg.solve(A.T @ A, A.T @ y_noisy)
c_ls_n = np.linalg.lstsq(A, y_noisy, rcond=None)[0]

tt = np.linspace(0, 1, 400)
Att = np.vander(tt, deg + 1, increasing=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.8))
ax1.plot(t, y_noisy, "ko", ms=4, label="data")
ax1.plot(tt, Att @ c_ls_n, "royalblue", lw=2, label="lstsq")
ax1.plot(tt, Att @ c_ne_n, "crimson", lw=1.5, ls="--", label="normal equations")
ax1.legend(); ax1.grid(alpha=0.3); ax1.set_title("fitted curves")
w = 0.4
ax2.bar(np.arange(deg + 1) - w / 2, c_ls_n, w, label="lstsq")
ax2.bar(np.arange(deg + 1) + w / 2, c_ne_n, w, label="normal equations")
ax2.set_yscale("symlog"); ax2.set_xlabel("power of t"); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_title("coefficients")
plt.show()
print(f"residual: lstsq {np.linalg.norm(A @ c_ls_n - y_noisy):.4f}   normal eq. {np.linalg.norm(A @ c_ne_n - y_noisy):.4f}")

### Verify numerically: the identity through the SVD

In [ ]:
s = np.linalg.svd(A, compute_uv=False)
s_AtA = np.linalg.svd(A.T @ A, compute_uv=False)
print(" i |      σᵢ(A)²      | σᵢ(AᵀA) computed")
for i in range(deg + 1):
    print(f"{i:>2} | {s[i]**2:.6e} | {s_AtA[i]:.6e}")

A5 = np.vander(t, 6, increasing=True)                 # a smaller degree, where everything is still accurate
check("degree 5: κ(AᵀA) = κ(A)²", np.isclose(np.linalg.cond(A5.T @ A5), np.linalg.cond(A5) ** 2, rtol=1e-4))
check("degree 10: the largest singular values of AᵀA are σᵢ(A)²", np.allclose(s_AtA[:4], s[:4] ** 2, rtol=1e-8))

### Explore: degree 1 to 15

In [ ]:
degs = np.arange(1, 16)
rows = []
for d in degs:
    Ad = np.vander(t, d + 1, increasing=True)
    cd = np.random.default_rng(d).standard_normal(d + 1)
    yd = Ad @ cd
    err = lambda c: np.linalg.norm(c - cd) / np.linalg.norm(cd)
    rows.append((np.linalg.cond(Ad), np.linalg.cond(Ad.T @ Ad),
                 err(np.linalg.solve(Ad.T @ Ad, Ad.T @ yd)), err(np.linalg.lstsq(Ad, yd, rcond=None)[0])))
rows = np.array(rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.8))
ax1.semilogy(degs, rows[:, 0], "o-", label="κ(A)")
ax1.semilogy(degs, rows[:, 1], "s-", label="κ(AᵀA) computed")
ax1.semilogy(degs, rows[:, 0] ** 2, "k--", label="κ(A)²")
ax1.axhline(1 / np.finfo(float).eps, color="gray", ls=":", label="1 / machine ε")
ax1.set_xlabel("degree"); ax1.legend(); ax1.grid(alpha=0.3, which="both"); ax1.set_title("condition numbers")
ax2.semilogy(degs, rows[:, 2], "o-", color="crimson", label="normal equations")
ax2.semilogy(degs, rows[:, 3], "o-", color="royalblue", label="lstsq")
ax2.set_xlabel("degree"); ax2.set_ylabel("relative coefficient error")
ax2.legend(); ax2.grid(alpha=0.3, which="both"); ax2.set_title("accuracy")
plt.show()

---
### Explain (hand-in — in your own words)

**1. Why does the identity $\kappa(A^\top A) = \kappa(A)^2$ hold (use the SVD)?**
*(Hint: write $A^\top A = V\Sigma^\top\Sigma V^\top$. What are its singular values?)*

*Your answer.*

**2. Why does it make the normal equations less accurate?**
*(Hint: a rule of thumb: solving a system with condition number $\kappa$ loses about $\log_{10}\kappa$ of the ~16 digits of double precision. Also: why does the computed $\kappa(A^\top A)$ stop at about $10^{16}$–$10^{18}$?)*

*Your answer.*